In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import (
    datasets,
    transforms,
    models
)

from torch.utils.data import (
    DataLoader
)

import os
import copy
import numpy as np

In [2]:
from torchvision import transforms


transform = transforms.Compose([

    # simula recortes tipo ROI
    transforms.RandomResizedCrop(
        224,
        scale=(0.75, 1.0)
    ),

    # espejo
    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    # inclinación cámara
    transforms.RandomRotation(
        25
    ),

    # iluminación real
    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1
    ),

    # zoom y movimiento
    transforms.RandomAffine(
        degrees=0,
        translate=(0.15, 0.15),
        scale=(0.8, 1.2)
    ),

    # perspectiva tipo webcam
    transforms.RandomPerspective(
        distortion_scale=0.25,
        p=0.4
    ),

    transforms.ToTensor(),

    # mano, sombras, objetos tapando
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.12)
    ),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [3]:
import os
import json

from torchvision import datasets
from torch.utils.data import (
    DataLoader,
    random_split
)


DATASET_PATH = "../data/food-101/images"


# =========================
# CARGAR DATASET
# =========================
dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transform
)


# =========================
# CLASES
# =========================
CLASSES = dataset.classes


os.makedirs(
    "../models",
    exist_ok=True
)


with open(
    "../models/classes.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        CLASSES,
        f
    )


# =========================
# TRAIN / VALIDATION
# =========================
train_size = int(
    0.8 * len(dataset)
)

val_size = (
    len(dataset)
    - train_size
)


train_dataset, val_dataset = random_split(

    dataset,

    [
        train_size,
        val_size
    ]
)


# =========================
# DATALOADERS
# =========================
train_loader = DataLoader(

    train_dataset,

    batch_size=64,

    shuffle=True,

    num_workers=4,

    pin_memory=True,

    persistent_workers=True
)


val_loader = DataLoader(

    val_dataset,

    batch_size=64,

    shuffle=False,

    num_workers=4,

    pin_memory=True,

    persistent_workers=True
)


print(
    "Total imágenes:",
    len(dataset)
)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Total clases:",
    len(CLASSES)
)

print(
    "Clases:",
    CLASSES
)

Total imágenes: 19545
Train: 15636
Validation: 3909
Total clases: 21
Clases: ['apple', 'banana', 'ceviche', 'chicken_wings', 'coca_cola', 'coffee', 'french_fries', 'fried_rice', 'hamburger', 'ice_cream', 'lemon', 'mango', 'monster_energy', 'nachos', 'pizza', 'ramen', 'spaghetti_bolognese', 'steak', 'tacos', 'water_bottle', 'watermelon']


In [4]:
import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)


OLD_MODEL_PATH = (
    "../models/modelo_efficientnet.pth"
)


# =========================
# CHECKPOINT ANTERIOR
# =========================
checkpoint = torch.load(

    OLD_MODEL_PATH,

    map_location="cpu",

    weights_only=True
)


old_classes = checkpoint[
    "classifier.1.weight"
].shape[0]


# =========================
# MODELO NUEVO
# =========================
model = efficientnet_b0(

    weights=(
        EfficientNet_B0_Weights.DEFAULT
    )
)


new_num_classes = len(
    CLASSES
)


new_classifier = nn.Linear(

    1280,

    new_num_classes
)


# =========================
# MODELO ANTERIOR
# =========================
old_model = efficientnet_b0(

    weights=None
)


old_model.classifier[1] = nn.Linear(

    1280,

    old_classes
)


old_model.load_state_dict(
    checkpoint
)


# =========================
# TRANSFER LEARNING
# =========================
with torch.no_grad():

    nn.init.xavier_uniform_(
        new_classifier.weight
    )


    # copiar conocimiento viejo
    new_classifier.weight[
        :old_classes
    ] = old_model.classifier[1].weight


    new_classifier.bias[
        :old_classes
    ] = old_model.classifier[1].bias


# nueva salida
model.classifier[1] = (
    new_classifier
)


# copiar features aprendidas
model.features.load_state_dict(

    old_model.features.state_dict()
)


# =========================
# FASE 1
# congelar excepto BN
# =========================
for param in model.parameters():

    param.requires_grad = False


# classifier sí aprende
for param in model.classifier.parameters():

    param.requires_grad = True


# BatchNorm sí aprende
for module in model.modules():

    if isinstance(
        module,
        nn.BatchNorm2d
    ):

        for param in module.parameters():

            param.requires_grad = True


print(
    "Modelo incremental listo"
)

print(
    "Clases antiguas:",
    old_classes
)

print(
    "Clases nuevas:",
    len(CLASSES)
)

Modelo incremental listo
Clases antiguas: 21
Clases nuevas: 21


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy


# =========================
# DEVICE
# =========================
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    device
)

print(
    "Usando:",
    device
)


criterion = nn.CrossEntropyLoss()


best_acc = 0.0

best_weights = copy.deepcopy(
    model.state_dict()
)


# =========================
# VALIDATION
# =========================
def evaluate():

    model.eval()

    correct = 0
    total = 0


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )


            outputs = model(
                images
            )


            preds = torch.argmax(
                outputs,
                dim=1
            )


            correct += (
                preds == labels
            ).sum().item()


            total += labels.size(
                0
            )


    return correct / total


# =========================
# FASE 1
# =========================
print(
    "Fase 1: Adaptando clases"
)


for param in model.parameters():

    param.requires_grad = False


for param in model.classifier.parameters():

    param.requires_grad = True


optimizer = optim.Adam(

    model.classifier.parameters(),

    lr=0.001
)


EPOCHS = 8


for epoch in range(
    EPOCHS
):

    model.train()

    total_loss = 0


    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad()


        outputs = model(
            images
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()

        optimizer.step()


        total_loss += (
            loss.item()
        )


    val_acc = evaluate()


    print(

        f"[F1] Epoch {epoch+1}/{EPOCHS} | "

        f"Loss: {total_loss:.4f} | "

        f"Val Acc: {val_acc:.4f}"
    )


# =========================
# FASE 2
# =========================
print(
    "Fase 2: Fine tuning"
)


for param in model.parameters():

    param.requires_grad = True


optimizer = optim.Adam(

    model.parameters(),

    lr=0.00003
)


EPOCHS = 20


patience = 5

no_improve = 0


for epoch in range(
    EPOCHS
):

    model.train()

    total_loss = 0


    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad()


        outputs = model(
            images
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()

        optimizer.step()


        total_loss += (
            loss.item()
        )


    val_acc = evaluate()


    print(

        f"[F2] Epoch {epoch+1}/{EPOCHS} | "

        f"Loss: {total_loss:.4f} | "

        f"Val Acc: {val_acc:.4f}"
    )


    # mejor modelo
    if val_acc > best_acc:

        best_acc = val_acc

        best_weights = copy.deepcopy(
            model.state_dict()
        )

        no_improve = 0


    else:

        no_improve += 1


    # early stop
    if no_improve >= patience:

        print(
            "Early stopping"
        )

        break


# =========================
# GUARDAR
# =========================
model.load_state_dict(
    best_weights
)


torch.save(

    model.state_dict(),

    "../models/modelo_efficientnet.pth"
)


print(
    "Modelo guardado"
)

print(
    "Best val acc:",
    best_acc
)

Usando: cuda
Fase 1: Adaptando clases
[F1] Epoch 1/8 | Loss: 27.7862 | Val Acc: 0.9803
[F1] Epoch 2/8 | Loss: 26.2221 | Val Acc: 0.9823
[F1] Epoch 3/8 | Loss: 24.5560 | Val Acc: 0.9808
[F1] Epoch 4/8 | Loss: 26.5086 | Val Acc: 0.9783
[F1] Epoch 5/8 | Loss: 26.9242 | Val Acc: 0.9821
[F1] Epoch 6/8 | Loss: 24.4910 | Val Acc: 0.9800
[F1] Epoch 7/8 | Loss: 25.1404 | Val Acc: 0.9859
[F1] Epoch 8/8 | Loss: 24.6106 | Val Acc: 0.9841
Fase 2: Fine tuning
[F2] Epoch 1/20 | Loss: 23.8641 | Val Acc: 0.9829
[F2] Epoch 2/20 | Loss: 20.5808 | Val Acc: 0.9839
[F2] Epoch 3/20 | Loss: 20.4440 | Val Acc: 0.9867
[F2] Epoch 4/20 | Loss: 21.0126 | Val Acc: 0.9844
[F2] Epoch 5/20 | Loss: 19.2162 | Val Acc: 0.9857
[F2] Epoch 6/20 | Loss: 20.0082 | Val Acc: 0.9798
[F2] Epoch 7/20 | Loss: 18.1507 | Val Acc: 0.9793
[F2] Epoch 8/20 | Loss: 18.3141 | Val Acc: 0.9823
Early stopping
Modelo guardado
Best val acc: 0.9866973650550013


In [6]:
import json
import os

CLASSES_PATH = "../models/classes.json"


# =========================
# SI YA EXISTEN CLASES:
# conservar las antiguas
# y agregar solo nuevas
# =========================
if os.path.exists(CLASSES_PATH):

    with open(
        CLASSES_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        old_classes = json.load(f)

else:

    old_classes = []


# unir sin duplicar y manteniendo orden
final_classes = old_classes.copy()

for c in CLASSES:

    if c not in final_classes:

        final_classes.append(c)


# guardar
with open(
    CLASSES_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_classes,
        f,
        ensure_ascii=False,
        indent=2
    )


CLASSES = final_classes


print("Classes actualizadas:", len(CLASSES))
print(CLASSES)

Classes actualizadas: 21
['apple', 'banana', 'ceviche', 'chicken_wings', 'coca_cola', 'coffee', 'french_fries', 'fried_rice', 'hamburger', 'ice_cream', 'lemon', 'mango', 'monster_energy', 'nachos', 'pizza', 'ramen', 'spaghetti_bolognese', 'steak', 'tacos', 'water_bottle', 'watermelon']
